# 03.a - Seleção de Features e Redução de Dimensionalidade

A base pós-integração continha 665 colunas. Modelos preditivos com dimensionalidade excessiva estão sujeitos a overfitting, multicolinearidade e perda de interpretabilidade clínica. Para mitigar esses riscos, foi aplicado um processo estruturado de redução de dimensionalidade em cinco etapas sequenciais:

1. **Remoção conceitual:** Variáveis com vazamento de dados (data leakage), identificadores únicos sem poder preditivo e variáveis redundantes. Ex: `motivo_saida`.
2. **Variáveis constantes:** Colunas nas quais todos os registros têm o mesmo valor.
3. **Variáveis quasi-constantes:** Colunas nas quais mais de 98% dos registros assumem o mesmo valor.
4. **Baixa correlação com o alvo:** Correlação de Pearson com `indicador_obito` inferior a |r| = 0,03.
5. **Alta ausência:** Proporção de ausência superior a 99% e textos redundantes.

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Paths
ROOT = Path("..").resolve()
DATA_PATH = ROOT / 'data' / 'processed' / 'base_modelagem.csv'
PROCESSED_PATH = ROOT / 'data' / 'processed' / 'base_modelagem_reduzida.csv'

# Config
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

In [2]:
# Carregamento da base
df = pd.read_csv(DATA_PATH, low_memory=False)

# Garantir que indicador_obito seja numérico
df['indicador_obito'] = pd.to_numeric(df['indicador_obito'], errors='coerce')

print(f"Formato inicial: {df.shape[0]} linhas e {df.shape[1]} colunas.")

Formato inicial: 380591 linhas e 663 colunas.


## Etapa 1: Remoção por critério conceitual
Removendo variáveis com vazamento de dados (data leakage), identificadores únicos sem poder preditivo e variáveis redundantes. Destaque para `motivo_saida`, cujo preenchimento equivale a antecipar o próprio desfecho.

In [3]:
# Colunas que representam leakage, identificadores ou data release posterior ao evento
leakage_keywords = [
    'motivo_saida', 'data_saida', 'cid_morte', 'numero_aih',
    'numero_remessa', 'sequencial', 'cnpj_hospital', 'cep_paciente',
    'cpf_gestor', 'cnpj_mantenedora', 'sequencial_remessa', 'cep_estabelecimento',
    'cpf_cnpj_estabelecimento', 'cnes_cnpj_mantenedora', 'arquivo_origem',
    'competencia', 'ano_competencia', 'mes_competencia', 'data_internacao',
    'ap01cv07', 'ap02cv07', 'ap03cv07', 'ap04cv07', 'ap05cv07', 'ap06cv07', 'ap07cv07', 'dt_atual'
]

cols_to_drop_step1 = []
for col in df.columns:
    col_lower = col.lower()
    if any(keyword in col_lower for keyword in leakage_keywords):
        cols_to_drop_step1.append(col)
    # Remover colunas sufixadas com DESC que são redundantes com os códigos
    if col_lower.endswith('_desc'):
        cols_to_drop_step1.append(col)

cols_to_drop_step1 = list(set(cols_to_drop_step1))
# Garantir que indicador_obito nunca seja removido acidentalmente
if 'indicador_obito' in cols_to_drop_step1:
    cols_to_drop_step1.remove('indicador_obito')

df.drop(columns=[c for c in cols_to_drop_step1 if c in df.columns], inplace=True)
print(f"Removidas {len(cols_to_drop_step1)} colunas na Etapa 1.")
print(f"Formato atual: {df.shape}")

Removidas 85 colunas na Etapa 1.
Formato atual: (380591, 578)


## Etapa 2: Remoção de variáveis constantes
Eliminação de variáveis numéricas constantes — aquelas com valor único em toda a base. Essas não possuem capacidade discriminativa.

In [4]:
constantes = [c for c in df.columns if df[c].nunique(dropna=True) <= 1]
df.drop(columns=constantes, inplace=True)
print(f"Removidas {len(constantes)} colunas constantes na Etapa 2.")
print(f"Formato atual: {df.shape}")

Removidas 93 colunas constantes na Etapa 2.
Formato atual: (380591, 485)


## Etapa 3: Remoção de variáveis quasi-constantes
Eliminação de variáveis quasi-constantes, definidas como colunas nas quais mais de 98% dos registros assumem o mesmo valor.

In [5]:
quasi_constantes = []
for c in df.columns:
    if c != 'indicador_obito':
        top_freq = df[c].value_counts(normalize=True, dropna=False).iloc[0]
        if top_freq > 0.98:
            quasi_constantes.append(c)

df.drop(columns=quasi_constantes, inplace=True)
print(f"Removidas {len(quasi_constantes)} colunas quasi-constantes (>98%) na Etapa 3.")
print(f"Formato atual: {df.shape}")

Removidas 14 colunas quasi-constantes (>98%) na Etapa 3.
Formato atual: (380591, 471)


## Etapa 4: Remoção de variáveis com baixa correlação
Remoção de variáveis numéricas com correlação de Pearson com a variável-alvo `indicador_obito` inferior a |r| = 0,03. (Limiar conservador).

In [6]:
# Selecionar colunas numéricas
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if 'indicador_obito' in num_cols:
    correlations = df[num_cols].corr()['indicador_obito'].abs()
    low_corr_cols = correlations[correlations < 0.03].index.tolist()
    
    if 'indicador_obito' in low_corr_cols:
        low_corr_cols.remove('indicador_obito')
        
    df.drop(columns=low_corr_cols, inplace=True)
    print(f"Removidas {len(low_corr_cols)} colunas com correlação < 0.03 na Etapa 4.")
else:
    print("Alvo não numérico ou não encontrado.")
    
print(f"Formato atual: {df.shape}")

Removidas 368 colunas com correlação < 0.03 na Etapa 4.
Formato atual: (380591, 103)


## Etapa 5: Alta ausência e textos redundantes
Exclusão de variáveis com proporção de ausência superior a 99% e variáveis textuais redundantes/não codificadas.

In [7]:
alta_ausencia = [c for c in df.columns if df[c].isna().mean() > 0.99]
df.drop(columns=alta_ausencia, inplace=True)
print(f"Removidas {len(alta_ausencia)} colunas com ausência > 99% na Etapa 5.")

# Textos redundantes ou com cardinalidade extrema (exceto codigo_cnes, que pode ser agrupado futuramente se desejado, mas aqui será limpo caso necessário).
# Para modelagem, vamos dropar objects não convertidos
text_cols = df.select_dtypes(include=['object']).columns.tolist()
cols_text_drop = [c for c in text_cols if df[c].nunique() > 100]
if 'codigo_cnes' in cols_text_drop:
    cols_text_drop.remove('codigo_cnes') # Manter cnes para referencial, se precisar

df.drop(columns=cols_text_drop, inplace=True)
print(f"Removidas {len(cols_text_drop)} colunas de texto puras/alta cardinalidade.")

print(f"Formato atual: {df.shape}")

Removidas 0 colunas com ausência > 99% na Etapa 5.
Removidas 4 colunas de texto puras/alta cardinalidade.
Formato atual: (380591, 99)


/tmp/ipykernel_200899/215987986.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include=['object']).columns.tolist()


## Salvamento da Base Reduzida

In [8]:
df.to_csv(PROCESSED_PATH, index=False)
print(f"Base reduzida salva em: {PROCESSED_PATH}")
print(f"Formato final: {df.shape[0]} linhas e {df.shape[1]} colunas.")

Base reduzida salva em: /home/carolina/Documents/TCC Documentos/TCC/data/processed/base_modelagem_reduzida.csv
Formato final: 380591 linhas e 99 colunas.
